# Data Preprocessing

This notebook prepares the project data for exploratory analysis, visualization, and modeling. The goal is to create a clean match-level dataset for FIFA World Cup match outcome prediction while avoiding data leakage.

## Dataset usage decisions

- **Primary modeling dataset:** `results.csv`
  - Contains one row per international match.
  - Includes match date, home team, away team, scores, tournament, city, country, and neutral-site status.
  - Serves as the base table because the model predicts outcomes at the match level.

- **Main external feature dataset:** `eloratings.csv`
  - Contains historical Elo ratings by team and date.
  - Elo ratings are merged onto the match results data to create pre-match team-strength features.
  - Main Elo-based features: `home_elo`, `away_elo`, and `elo_diff`.

- **Datasets reviewed but not merged into the first modeling dataset:**
  - `matches.csv`, `teams.csv`, `tournaments.csv`, and `tournament_standings.csv` contain useful World Cup metadata for EDA or later feature engineering.
  - Some columns in these files describe post-match or post-tournament outcomes, such as match result, score margin, tournament winner, final standing, extra time, and penalty shootout information. These would not be known before kickoff and could introduce data leakage if used as predictors.
  - `goalscorers.csv` and `shootouts.csv` are event-level datasets that describe events during or after a match. They may be useful later if aggregated carefully using only information available before each match date, but they are excluded from the first modeling dataset.

- **Output files created by this notebook:**
  - `matches_preprocessed_readable.csv`: readable match-level dataset for EDA and visualization.
  - `matches_modeling_base.csv`: modeling dataset with selected pre-match features before encoding/scaling.
  - `X_train_processed.csv`, `X_test_processed.csv`, `y_train.csv`, and `y_test.csv`: encoded/scaled train/test files for modeling.
  - `feature_names.csv` and `label_mapping.csv`: metadata to help interpret processed model inputs and encoded target labels.


## Imports and Data Loading

The notebook loads all raw datasets so that their structures can be reviewed, but the first modeling pipeline uses `results.csv` and `eloratings.csv` as the main inputs.


In [55]:
import pandas as pd
import numpy as np
from pathlib import Path

# Use ../data when running from a notebooks/ folder; fall back to ./data when running from the repo root.
DATA_DIR = Path("../data")
if not (DATA_DIR / "results.csv").exists():
    DATA_DIR = Path("./data")

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Using data directory:", DATA_DIR.resolve())
print("Processed outputs will be saved to:", PROCESSED_DIR.resolve())


Using data directory: /Users/oliviadarby/DATASCI_207/ucb_mids_207_world_cup_prediction_model/data
Processed outputs will be saved to: /Users/oliviadarby/DATASCI_207/ucb_mids_207_world_cup_prediction_model/data/processed


In [75]:
# Fjelstul World Cup Database files
matches = pd.read_csv(DATA_DIR / "matches.csv")
teams = pd.read_csv(DATA_DIR / "teams.csv")
tournaments = pd.read_csv(DATA_DIR / "tournaments.csv")
standings = pd.read_csv(DATA_DIR / "tournament_standings.csv")

# International Football Results files (using new files with 2026 World Cup
results = pd.read_csv(DATA_DIR / "results_new.csv")
goalscorers = pd.read_csv(DATA_DIR / "goalscorers_new.csv")
shootouts = pd.read_csv(DATA_DIR / "shootouts_new.csv")

# Elo ratings file
eloratings = pd.read_csv(DATA_DIR / "eloratings.csv")


In [76]:
# Check the temporal coverage of the updated data sources.
results_dates_check = pd.to_datetime(
    results["date"],
    format="mixed",
    errors="coerce"
)

elo_dates_check = pd.to_datetime(
    eloratings["date"],
    format="mixed",
    errors="coerce"
)

print("Results coverage:", results_dates_check.min(), "to", results_dates_check.max())
print("Elo coverage:", elo_dates_check.min(), "to", elo_dates_check.max())

Results coverage: 1872-11-30 00:00:00 to 2026-07-19 00:00:00
Elo coverage: 1872-11-30 00:00:00 to 2025-12-13 00:00:00


In [77]:
# Structure check
dataframes = {
    "matches": matches,
    "teams": teams,
    "tournaments": tournaments,
    "standings": standings,
    "results": results,
    "shootouts": shootouts,
    "goalscorers": goalscorers,
    "eloratings": eloratings
}

for name, df in dataframes.items():
    print(f"{name}: {df.shape}")
    display(df.head())


matches: (1248, 37)


,key_id,tournament_id,tournament_name,match_id,match_name,stage_name,group_name,group_stage,knockout_stage,replayed,...,away_team_score_margin,extra_time,penalty_shootout,score_penalties,home_team_score_penalties,away_team_score_penalties,result,home_team_win,away_team_win,draw
0,1,WC-1930,1930 FIFA Men's World Cup,M-1930-01,France vs Mexico,group stage,Group 1,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
1,2,WC-1930,1930 FIFA Men's World Cup,M-1930-02,United States vs Belgium,group stage,Group 4,1,0,0,...,-3,0,0,0-0,0,0,home team win,1,0,0
2,3,WC-1930,1930 FIFA Men's World Cup,M-1930-03,Yugoslavia vs Brazil,group stage,Group 2,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0
3,4,WC-1930,1930 FIFA Men's World Cup,M-1930-04,Romania vs Peru,group stage,Group 3,1,0,0,...,-2,0,0,0-0,0,0,home team win,1,0,0
4,5,WC-1930,1930 FIFA Men's World Cup,M-1930-05,Argentina vs France,group stage,Group 1,1,0,0,...,-1,0,0,0-0,0,0,home team win,1,0,0


teams: (88, 14)


,key_id,team_id,team_name,team_code,mens_team,womens_team,federation_name,region_name,confederation_id,confederation_name,confederation_code,mens_team_wikipedia_link,womens_team_wikipedia_link,federation_wikipedia_link
0,1,T-01,Algeria,DZA,1,0,Algerian Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Algeria_national...,not applicable,https://en.wikipedia.org/wiki/Algerian_Footbal...
1,2,T-02,Angola,AGO,1,0,Angolan Football Federation,Africa,CF-2,Confederation of African Football,CAF,https://en.wikipedia.org/wiki/Angola_national_...,not applicable,https://en.wikipedia.org/wiki/Angolan_Football...
2,3,T-03,Argentina,ARG,1,1,Argentine Football Association,South America,CF-4,South American Football Confederation,CONMEBOL,https://en.wikipedia.org/wiki/Argentina_nation...,https://en.wikipedia.org/wiki/Argentina_women'...,https://en.wikipedia.org/wiki/Argentine_Footba...
3,4,T-04,Australia,AUS,1,1,Football Australia,Oceania,CF-1,Asian Football Confederation,AFC,https://en.wikipedia.org/wiki/Australia_men%27...,https://en.wikipedia.org/wiki/Australia_women'...,https://en.wikipedia.org/wiki/Football_Australia
4,5,T-05,Austria,AUT,1,0,Austrian Football Association,Europe,CF-6,Union of European Football Associations,UEFA,https://en.wikipedia.org/wiki/Austria_national...,not applicable,https://en.wikipedia.org/wiki/Austrian_Footbal...


tournaments: (30, 18)


...

,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN
3,1972-05-17,Thailand,South Korea,South Korea,NaN
4,1972-05-19,Thailand,Cambodia,Thailand,NaN


goalscorers: (47903, 8)


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False
3,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,75.0,False,False
4,1916-07-06,Argentina,Chile,Argentina,Alberto Ohaco,2.0,False,False


eloratings: (6678, 4)


,date,team,rating,change
0,1872-11-30,England,2003.0,3
1,1872-11-30,Scotland,1997.0,-3
2,1873-03-08,England,2014.0,11
3,1873-03-08,Scotland,1986.0,-11
4,1874-03-07,England,2006.0,-8


## Missing Value Handling and Basic Cleaning

The main supervised-learning target requires completed matches. Rows without final home or away scores are removed because their outcomes are not known. Dates are also converted to datetime format so the Elo merge and time-based split work correctly.


In [78]:
# Convert date columns.
# Both datasets contain mixed date formats.
results["date"] = pd.to_datetime(
    results["date"],
    format="mixed",
    errors="coerce"
)

eloratings["date"] = pd.to_datetime(
    eloratings["date"],
    format="mixed",
    errors="coerce"
)

print("Missing results dates:", results["date"].isna().sum())
print("Missing Elo dates:", eloratings["date"].isna().sum())

print("Latest match date:", results["date"].max())
print("Latest Elo date:", eloratings["date"].max())

Missing results dates: 0
Missing Elo dates: 0
Latest match date: 2026-07-19 00:00:00
Latest Elo date: 2025-12-13 00:00:00


In [79]:
# Examine incomplete matches before removing them.
missing_scores = results.loc[
    results["home_score"].isna() |
    results["away_score"].isna()
].copy()

print("Rows with missing home or away score:", missing_scores.shape[0])

# Check whether any incomplete rows are 2026 World Cup matches.
missing_wc_2026 = missing_scores.loc[
    (missing_scores["date"].dt.year == 2026) &
    (missing_scores["tournament"] == "FIFA World Cup")
]

print(
    "Incomplete 2026 World Cup rows:",
    missing_wc_2026.shape[0]
)

display(
    missing_wc_2026[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score"
        ]
    ]
)

# Keep only rows with valid dates and completed scores.
results = results.dropna(
    subset=["date", "home_score", "away_score"]
).copy()

results["home_score"] = results["home_score"].astype(int)
results["away_score"] = results["away_score"].astype(int)

print("Results shape after cleaning:", results.shape)
print("Duplicate rows:", results.duplicated().sum())

Rows with missing home or away score: 2
Incomplete 2026 World Cup rows: 2


,date,home_team,away_team,home_score,away_score
49518,2026-07-18,France,England,NaN,NaN
49519,2026-07-19,Spain,Argentina,NaN,NaN


Results shape after cleaning: (49518, 9)
Duplicate rows: 0


The most recently updated results file doesn't include the final scores for 2026 final and third-place matches, so they were removed during cleaning.

## Target Variable and Initial Feature Engineering

The target variable is `match_result`, with three classes: `home_win`, `away_win`, and `draw`. Score-based columns are created for EDA and validation, but they are later excluded from the modeling feature set because they are only known after the match.


In [80]:
def get_match_result(row):
    if row["home_score"] > row["away_score"]:
        return "home_win"
    elif row["home_score"] < row["away_score"]:
        return "away_win"
    return "draw"


results["match_result"] = results.apply(
    get_match_result,
    axis=1
)

# Score-based features are retained for EDA only.
results["goal_diff"] = (
    results["home_score"] -
    results["away_score"]
)

results["abs_goal_diff"] = results["goal_diff"].abs()

results["total_goals"] = (
    results["home_score"] +
    results["away_score"]
)

results["year"] = results["date"].dt.year

# Exact matching prevents World Cup qualification matches from being included.
results["is_world_cup"] = (
    results["tournament"] == "FIFA World Cup"
)

print("Target distribution:")
print(results["match_result"].value_counts())

print("\n2026 tournament counts:")
print(
    results.loc[
        results["year"] == 2026,
        "tournament"
    ].value_counts()
)

available_wc_2026 = results.loc[
    (results["year"] == 2026) &
    (results["is_world_cup"])
]

print(
    "\nCompleted 2026 World Cup matches available:",
    available_wc_2026.shape[0]
)

Target distribution:
match_result
home_win    24264
away_win    13996
draw        11258
Name: count, dtype: int64

2026 tournament counts:
tournament
Friendly                                             207
FIFA World Cup                                       102
FIFA Series                                           33
African Cup of Nations                                16
FIFA World Cup qualification                          16
CONCACAF Series                                       16
CONIFA European Football Cup                           9
Morocco, Capital of African Football                   6
Unity Cup                                              4
Diamond Jubilee International Football Tournament      4
Baltic Cup                                             4
Mukuru 4 Nations                                       2
Tri-Nations Cup                                        2
Name: count, dtype: int64

Completed 2026 World Cup matches available: 102


## Outlier Detection and Treatment

Scoreline outliers are checked using absolute goal differential. Extreme results are inspected rather than automatically removed. Because these appear to be legitimate historical matches rather than data-entry errors, they are retained.


In [81]:
results["abs_goal_diff"].describe()


count    49518.000000
mean         1.717335
std          1.793140
min          0.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         31.000000
Name: abs_goal_diff, dtype: float64

In [82]:
extreme_scorelines = results[results["abs_goal_diff"] >= 20].copy()

print("Number of matches with absolute goal differential >= 20:", extreme_scorelines.shape[0])

extreme_scorelines[[
    "date", "home_team", "away_team", "home_score", "away_score", "tournament", "abs_goal_diff"
]].sort_values("abs_goal_diff", ascending=False)


Number of matches with absolute goal differential >= 20: 13


,date,home_team,away_team,home_score,away_score,tournament,abs_goal_diff
25425,2001-04-11,Australia,American Samoa,31,0,FIFA World Cup qualification,31
8551,1971-09-13,Tahiti,Cook Islands,30,0,South Pacific Games,30
11916,1979-08-30,Fiji,Kiribati,24,0,South Pacific Games,24
25422,2001-04-09,Australia,Tonga,22,0,FIFA World Cup qualification,22
37064,2013-06-24,Provence,Tibet,22,0,"International Tournament of Peoples, Cultures ...",22
6580,1966-04-03,Libya,Oman,21,0,Arab Cup,21
29045,2005-03-11,Guam,North Korea,0,21,EAFF Championship,21
37062,2013-06-23,Quebec,Tibet,21,0,"International Tournament of Peoples, Cultures ...",21
15922,1987-12-15,American Samoa,Papua New Guinea,0,20,South Pacific Games,20
24181,2000-02-14,Kuwait,Bhutan,20,0,AFC Asian Cup qualification,20


## Team Name Standardization

Team names must be standardized before merging match results with Elo ratings. Manual mappings are applied only for clear naming differences between the two datasets. Remaining unmatched teams are reviewed; most are regional, non-FIFA, or representative teams and are not manually forced into a national-team mapping.


In [83]:
# Clean hidden/non-breaking spaces in Elo team names.
eloratings["team"] = (
    eloratings["team"]
    .str.replace("\xa0", " ", regex=False)
    .str.strip()
)

# Candidate mappings for common naming differences.
POSSIBLE_TEAM_NAME_MAPPING = {
    "Czech Republic": "Czechia",
    "Republic of Ireland": "Ireland",
    "Chinese Taipei": "Taiwan",
    "USA": "United States",
    "United States of America": "United States",
    "DR Congo": "Democratic Republic of Congo",
    "Democratic Republic of the Congo": "Democratic Republic of Congo",
    "German DR": "East Germany",
    "North Macedonia": "Macedonia",
    "FYR Macedonia": "Macedonia",
    "Eswatini": "Swaziland",
    "Myanmar": "Burma",
    "Macau": "Macao",
    "Timor-Leste": "East Timor",
    "Cabo Verde": "Cape Verde",
    "Côte d'Ivoire": "Ivory Coast",
    "Cote d'Ivoire": "Ivory Coast",
    "Türkiye": "Turkey",
    "Curaçao": "Curacao",
    "São Tomé and Príncipe": "Sao Tome and Principe",
    "São Tomé and Principe": "Sao Tome and Principe",
    "Sao Tome and Príncipe": "Sao Tome and Principe",
    "St. Kitts and Nevis": "Saint Kitts and Nevis",
    "St Kitts and Nevis": "Saint Kitts and Nevis",
    "St. Lucia": "Saint Lucia",
    "St Lucia": "Saint Lucia",
    "St. Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "St Vincent and the Grenadines": "Saint Vincent and the Grenadines",
    "United States Virgin Islands": "US Virgin Islands",
    "Kyrgyz Republic": "Kyrgyzstan",
    "Brunei Darussalam": "Brunei",
    "Lao": "Laos",
    "Lao PDR": "Laos"
}

# Keep only mappings whose target names actually appear in the Elo file.
elo_team_names = set(eloratings["team"].dropna().unique())
TEAM_NAME_MAPPING = {
    old: new
    for old, new in POSSIBLE_TEAM_NAME_MAPPING.items()
    if new in elo_team_names
}

print("Mappings used:")
TEAM_NAME_MAPPING


Mappings used:


{'Czech Republic': 'Czechia',
 'Republic of Ireland': 'Ireland',
 'Chinese Taipei': 'Taiwan',
 'USA': 'United States',
 'United States of America': 'United States',
 'DR Congo': 'Democratic Republic of Congo',
 'Democratic Republic of the Congo': 'Democratic Republic of Congo',
 'German DR': 'East Germany',
 'North Macedonia': 'Macedonia',
 'FYR Macedonia': 'Macedonia',
 'Eswatini': 'Swaziland',
 'Myanmar': 'Burma',
 'Macau': 'Macao',
 'Timor-Leste': 'East Timor',
 'Cabo Verde': 'Cape Verde',
 "Côte d'Ivoire": 'Ivory Coast',
 "Cote d'Ivoire": 'Ivory Coast',
 'Türkiye': 'Turkey',
 'São Tomé and Príncipe': 'Sao Tome and Principe',
 'São Tomé and Principe': 'Sao Tome and Principe',
 'Sao Tome and Príncipe': 'Sao Tome and Principe',
 'St. Kitts and Nevis': 'Saint Kitts and Nevis',
 'St Kitts and Nevis': 'Saint Kitts and Nevis',
 'St. Lucia': 'Saint Lucia',
 'St Lucia': 'Saint Lucia',
 'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
 'St Vincent and the Grenadines': 'S

In [84]:
results["home_team_clean"] = results["home_team"].replace(TEAM_NAME_MAPPING)
results["away_team_clean"] = results["away_team"].replace(TEAM_NAME_MAPPING)
eloratings["team_clean"] = eloratings["team"].replace(TEAM_NAME_MAPPING)

results_teams = set(results["home_team_clean"]).union(set(results["away_team_clean"]))
elo_teams = set(eloratings["team_clean"])

unmatched_results_teams = sorted(results_teams - elo_teams)
print("Number of teams in results not found in Elo:", len(unmatched_results_teams))
unmatched_results_teams[:50]


Number of teams in results not found in Elo: 98


['Abkhazia',
 'Alderney',
 'Ambazonia',
 'American Samoa',
 'Andalusia',
 'Arameans Suryoye',
 'Artsakh',
 'Asturias',
 'Aymara',
 'Barawa',
 'Basque Country',
 'Biafra',
 'Brittany',
 'Canary Islands',
 'Canton Ticino',
 'Cascadia',
 'Catalonia',
 'Central Spain',
 'Chameria',
 'Chechnya',
 'Cilento',
 'Corsica',
 'County of Nice',
 'Crimea',
 'Darfur',
 'Donetsk PR',
 'Délvidék',
 'East Turkestan',
 'Elba Island',
 'Ellan Vannin',
 'Felvidék',
 'Franconia',
 'Frøya',
 'Galicia',
 'Gotland',
 'Gozo',
 'Guernsey',
 'Găgăuzia',
 'Hitra',
 'Hmong',
 'Iraqi Kurdistan',
 'Isle of Man',
 'Isle of Wight',
 'Jersey',
 'Kabylia',
 'Kernow',
 'Kárpátalja',
 'Luhansk PR',
 'Madrid',
 'Manchukuo']

In [66]:
missing_elo_team_set = set(unmatched_results_teams)

rows_with_unmatched_team = results[
    results["home_team_clean"].isin(missing_elo_team_set) |
    results["away_team_clean"].isin(missing_elo_team_set)
]

print("Rows involving unmatched teams:", rows_with_unmatched_team.shape[0])
print("Percent of results data:", round(100 * rows_with_unmatched_team.shape[0] / results.shape[0], 2))

unmatched_home_counts = results.loc[
    results["home_team_clean"].isin(missing_elo_team_set),
    "home_team_clean"
].value_counts()

unmatched_away_counts = results.loc[
    results["away_team_clean"].isin(missing_elo_team_set),
    "away_team_clean"
].value_counts()

unmatched_counts = unmatched_home_counts.add(unmatched_away_counts, fill_value=0).sort_values(ascending=False)
unmatched_counts.head(30)


Rows involving unmatched teams: 1622
Percent of results data: 3.28


Guernsey                     240.0
Jersey                       235.0
Vietnam Republic             195.0
Alderney                     135.0
Réunion                      124.0
Ynys Môn                      67.0
Basque Country                64.0
Isle of Man                   58.0
Shetland                      58.0
American Samoa                55.0
Åland Islands                 51.0
Isle of Wight                 48.0
Catalonia                     48.0
Padania                       48.0
Frøya                         37.0
Saare County                  35.0
Western Isles                 34.0
Occitania                     33.0
Abkhazia                      33.0
Western Australia             32.0
Sápmi                         31.0
Gotland                       30.0
Raetia                        29.0
Orkney                        28.0
Tamil Eelam                   28.0
Iraqi Kurdistan               27.0
Székely Land                  27.0
Yemen DPR                     25.0
Menorca             

## Merge Home and Away Elo Ratings

Elo ratings are merged using the most recent rating available **before** each match date. `allow_exact_matches=False` is used because ratings recorded on the same date as a match may already include the result of that match. This avoids data leakage.

### Elo Availability and 2026 Test Design

The match-results dataset currently extends through July 15, 2026, while the Elo dataset ends on December 13, 2025. Consequently, each 2026 World Cup match is assigned the team's most recent Elo rating available before 2026. These ratings remain effectively frozen throughout the held-out test period. This creates a fully prospective evaluation: no information from matches played during 2026 is incorporated into the Elo predictors used to evaluate the 2026 World Cup. The downloaded match-results snapshot contains 102 completed FIFA World Cup matches through July 15, 2026. The third-place match and final are not included in this version of the source data.


In [85]:
print("Latest cleaned match date:", results["date"].max())
print("Latest cleaned Elo date:", eloratings["date"].max())

ELO_CUTOFF_DATE = pd.Timestamp("2026-01-01")

elo_pre_2026 = eloratings.loc[
    eloratings["date"] < ELO_CUTOFF_DATE
].copy()

print(
    "Latest Elo date used for the 2026 test period:",
    elo_pre_2026["date"].max()
)

wc_2026_teams = set(
    results.loc[
        (results["year"] == 2026) &
        (results["is_world_cup"]),
        "home_team_clean"
    ]
).union(
    set(
        results.loc[
            (results["year"] == 2026) &
            (results["is_world_cup"]),
            "away_team_clean"
        ]
    )
)

teams_with_pre_2026_elo = set(
    elo_pre_2026["team_clean"].dropna()
)

missing_pre_2026_elo_teams = sorted(
    wc_2026_teams - teams_with_pre_2026_elo
)

print(
    "2026 World Cup teams without any pre-2026 Elo:",
    missing_pre_2026_elo_teams
)

Latest cleaned match date: 2026-07-15 00:00:00
Latest cleaned Elo date: 2025-12-13 00:00:00
Latest Elo date used for the 2026 test period: 2025-12-13 00:00:00
2026 World Cup teams without any pre-2026 Elo: []


In [86]:
# Sort dataframes before merge_asof.
results_sorted = results.sort_values("date").copy()

elo_sorted = (
    eloratings[
        ["date", "team_clean", "rating"]
    ]
    .dropna(subset=["date", "team_clean", "rating"])
    .sort_values("date")
    .copy()
)

# Merge home-team Elo.
home_merged = pd.merge_asof(
    results_sorted,
    elo_sorted,
    left_on="date",
    right_on="date",
    left_by="home_team_clean",
    right_by="team_clean",
    direction="backward",
    allow_exact_matches=False
)

home_merged = (
    home_merged
    .rename(columns={"rating": "home_elo"})
    .drop(columns=["team_clean"])
)

# Merge away-team Elo.
model_df = pd.merge_asof(
    home_merged.sort_values("date"),
    elo_sorted,
    left_on="date",
    right_on="date",
    left_by="away_team_clean",
    right_by="team_clean",
    direction="backward",
    allow_exact_matches=False
)

model_df = (
    model_df
    .rename(columns={"rating": "away_elo"})
    .drop(columns=["team_clean"])
)

model_df["elo_diff"] = (
    model_df["home_elo"] -
    model_df["away_elo"]
)

model_df.head()


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,match_result,goal_diff,abs_goal_diff,total_goals,year,is_world_cup,home_team_clean,away_team_clean,home_elo,away_elo,elo_diff
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,draw,0,0,0,1872,False,Scotland,England,NaN,NaN,NaN
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,home_win,2,2,6,1873,False,England,Scotland,2003.0,1997.0,6.0
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,home_win,1,1,3,1874,False,Scotland,England,1986.0,2014.0,-28.0
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,draw,0,0,4,1875,False,England,Scotland,2006.0,1994.0,12.0
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,home_win,3,3,3,1876,False,Scotland,England,1997.0,2003.0,-6.0


In [87]:
missing_elo_rows = model_df.loc[
    model_df["home_elo"].isna() |
    model_df["away_elo"].isna()
]

print("Rows after Elo merge:", model_df.shape[0])
print(
    "Rows missing at least one Elo rating:",
    missing_elo_rows.shape[0]
)

print(
    "Percent missing Elo:",
    round(
        100 * missing_elo_rows.shape[0] /
        model_df.shape[0],
        2
    )
)

print("\nMissing Elo values by column:")
print(
    model_df[
        ["home_elo", "away_elo", "elo_diff"]
    ].isna().sum()
)

Rows after Elo merge: 49518
Rows missing at least one Elo rating: 7459
Percent missing Elo: 15.06

Missing Elo values by column:
home_elo    4701
away_elo    5060
elo_diff    7459
dtype: int64


In [88]:
# Actual FIFA World Cup tournament matches only.
world_cup_df = model_df.loc[
    model_df["tournament"] == "FIFA World Cup"
].copy()

wc_missing_elo = world_cup_df.loc[
    world_cup_df["home_elo"].isna() |
    world_cup_df["away_elo"].isna()
]

print("FIFA World Cup rows:", world_cup_df.shape[0])

print(
    "FIFA World Cup rows missing at least one Elo:",
    wc_missing_elo.shape[0]
)

print(
    "Percent FIFA World Cup missing Elo:",
    round(
        100 * wc_missing_elo.shape[0] /
        world_cup_df.shape[0],
        2
    )
)

FIFA World Cup rows: 1066
FIFA World Cup rows missing at least one Elo: 83
Percent FIFA World Cup missing Elo: 7.79


In [89]:
# 2026 World Cup matches only
wc_2026_elo_check = model_df.loc[
    (model_df["year"] == 2026) &
    (model_df["tournament"] == "FIFA World Cup")
].copy()

print(
    "Available 2026 World Cup rows after Elo merge:",
    wc_2026_elo_check.shape[0]
)

print("\nMissing 2026 World Cup Elo values:")
print(
    wc_2026_elo_check[
        ["home_elo", "away_elo", "elo_diff"]
    ].isna().sum()
)

display(
    wc_2026_elo_check.loc[
        wc_2026_elo_check["home_elo"].isna() |
        wc_2026_elo_check["away_elo"].isna(),
        [
            "date",
            "home_team",
            "away_team",
            "home_team_clean",
            "away_team_clean"
        ]
    ]
)

Available 2026 World Cup rows after Elo merge: 102

Missing 2026 World Cup Elo values:
home_elo    0
away_elo    0
elo_diff    0
dtype: int64


,date,home_team,away_team,home_team_clean,away_team_clean


In [90]:
# Compare possible year restrictions by Elo coverage.
coverage_summary = []

for cutoff in [1950, 1970, 1990, 1994, 2000, 2010]:
    subset = model_df[model_df["year"] >= cutoff]
    missing = subset[
        subset["home_elo"].isna() | subset["away_elo"].isna()
    ]
    coverage_summary.append({
        "cutoff_year": cutoff,
        "rows": subset.shape[0],
        "missing_elo_rows": missing.shape[0],
        "missing_elo_percent": round(100 * missing.shape[0] / subset.shape[0], 2),
        "usable_elo_rows": subset.dropna(subset=["home_elo", "away_elo", "elo_diff"]).shape[0]
    })

coverage_summary_df = pd.DataFrame(coverage_summary)
coverage_summary_df


,cutoff_year,rows,missing_elo_rows,missing_elo_percent,usable_elo_rows
0,1950,46180,5777,12.51,40403
1,1970,41558,4108,9.88,37450
2,1990,32400,2665,8.23,29735
3,1994,30015,2307,7.69,27708
4,2000,25456,1715,6.74,23741
5,2010,15927,915,5.74,15012


## Year Restriction and Evaluation Periods

The modeling dataset is restricted to matches from **1994 onward** to better represent modern international soccer while preserving a large training sample and improving Elo coverage.

The resulting data is divided chronologically:

- **Training:** international matches from 1994 through 2021
- **Validation:** international matches from 2022 through 2025
- **Final test:** completed 2026 FIFA World Cup matches available in the current data snapshot

The current results file extends through July 15, 2026 and includes 102 completed World Cup matches. The final and third-place match are not present in this version of the source.

The validation data is used for model selection and tuning. The 2026 World Cup observations remain fully held out until final evaluation.

In [91]:
START_YEAR = 1994

model_df_modern = model_df.loc[
    model_df["year"] >= START_YEAR
].copy()

missing_elo_modern = model_df_modern.loc[
    model_df_modern["home_elo"].isna() |
    model_df_modern["away_elo"].isna()
]

# Elo-based models require ratings for both teams.
model_elo_df = model_df_modern.dropna(
    subset=["home_elo", "away_elo", "elo_diff"]
).copy()

print("Original dataset shape:", model_df.shape)
print("Modern dataset shape:", model_df_modern.shape)

print(
    "Modern rows missing at least one Elo:",
    missing_elo_modern.shape[0]
)

print(
    "Modern missing-Elo percentage:",
    round(
        100 * missing_elo_modern.shape[0] /
        model_df_modern.shape[0],
        2
    )
)

print("Final Elo modeling rows:", model_elo_df.shape[0])

usable_wc_2026_count = model_elo_df.loc[
    (model_elo_df["year"] == 2026) &
    (model_elo_df["tournament"] == "FIFA World Cup")
].shape[0]

print(
    "Usable 2026 World Cup test matches:",
    usable_wc_2026_count
)

Original dataset shape: (49518, 20)
Modern dataset shape: (30015, 20)
Modern rows missing at least one Elo: 2307
Modern missing-Elo percentage: 7.69
Final Elo modeling rows: 27708
Usable 2026 World Cup test matches: 102


## Save Readable EDA Dataset

This dataset keeps human-readable columns and retains rows with missing Elo values. It is intended for EDA and visualization, not final model fitting.


In [92]:
readable_cols = [
    "date",
    "year",
    "home_team",
    "away_team",
    "home_team_clean",
    "away_team_clean",
    "home_score",
    "away_score",
    "match_result",
    "tournament",
    "city",
    "country",
    "neutral",
    "is_world_cup",
    "home_elo",
    "away_elo",
    "elo_diff",
    "goal_diff",
    "abs_goal_diff",
    "total_goals"
]

eda_df = model_df_modern[readable_cols].copy()

world_cup_2026_readable = eda_df.loc[
    (eda_df["year"] == 2026) &
    (eda_df["tournament"] == "FIFA World Cup")
].copy()

eda_df.to_csv(
    PROCESSED_DIR / "matches_preprocessed_readable.csv",
    index=False
)

world_cup_2026_readable.to_csv(
    PROCESSED_DIR /
    "world_cup_2026_preprocessed_readable.csv",
    index=False
)

print("Readable EDA dataset shape:", eda_df.shape)

print(
    "2026 World Cup readable dataset shape:",
    world_cup_2026_readable.shape
)

Readable EDA dataset shape: (30015, 20)
2026 World Cup readable dataset shape: (102, 20)


## Feature Selection

Only variables known before kickoff are used as predictors.

The baseline feature set consists of:

- `elo_diff`: the home team's pre-match Elo minus the away team's pre-match Elo
- `neutral`: whether the match was played at a neutral venue
- `is_world_cup`: whether the match was part of the FIFA World Cup
- `tournament`: the competition category

Score-based variables are retained for EDA but excluded from modeling because they reveal information observed only after the match.

Because `elo_diff` is calculated directly from `home_elo` and `away_elo`, the baseline model uses `elo_diff` alone rather than including all three highly related Elo variables.

In [93]:
target_col = "match_result"

numeric_features = [
    "elo_diff"
]

categorical_features = [
    "tournament",
    "neutral",
    "is_world_cup"
]

# Date and year are used for splitting and traceability,
# but are not passed to the model.
modeling_cols = (
    numeric_features +
    categorical_features +
    [target_col, "date", "year"]
)

modeling_base_df = model_elo_df[
    modeling_cols
].copy()

modeling_base_df.to_csv(
    PROCESSED_DIR / "matches_modeling_base.csv",
    index=False
)

print(
    "Modeling base dataset shape:",
    modeling_base_df.shape
)

modeling_base_df.head()

Modeling base dataset shape: (27708, 7)


,elo_diff,tournament,neutral,is_world_cup,match_result,date,year
19503,-115.0,Friendly,False,False,draw,1994-01-02,1994
19504,-41.0,Friendly,False,False,home_win,1994-01-02,1994
19505,177.0,Friendly,False,False,draw,1994-01-05,1994
19506,-424.0,Friendly,False,False,away_win,1994-01-09,1994
19507,-268.0,Friendly,False,False,draw,1994-01-11,1994


## Chronological Train, Validation, and Test Split

The data is separated chronologically to simulate future prediction:

- **Training set:** matches from 1994–2021
- **Validation set:** matches from 2022–2025
- **Test set:** completed matches from the 2026 FIFA World Cup

The validation set may be used to select hyperparameters, model settings, and heuristic thresholds. The 2026 World Cup test set must not be used for any tuning decisions.

In [94]:
train_df = modeling_base_df.loc[
    (modeling_base_df["year"] >= 1994) &
    (modeling_base_df["year"] < 2022)
].copy()

validation_df = modeling_base_df.loc[
    (modeling_base_df["year"] >= 2022) &
    (modeling_base_df["year"] < 2026)
].copy()

test_df = modeling_base_df.loc[
    (modeling_base_df["year"] == 2026) &
    (modeling_base_df["tournament"] == "FIFA World Cup")
].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

print(
    "Train years:",
    train_df["year"].min(),
    "-",
    train_df["year"].max()
)

print(
    "Validation years:",
    validation_df["year"].min(),
    "-",
    validation_df["year"].max()
)

print(
    "Test years:",
    test_df["year"].min(),
    "-",
    test_df["year"].max()
)

print("\nTraining target distribution:")
display(
    train_df[target_col]
    .value_counts(normalize=True)
    .rename("proportion")
)

print("\nValidation target distribution:")
display(
    validation_df[target_col]
    .value_counts(normalize=True)
    .rename("proportion")
)

print("\n2026 World Cup test target distribution:")
display(
    test_df[target_col]
    .value_counts(normalize=True)
    .rename("proportion")
)

EXPECTED_AVAILABLE_TEST_MATCHES = (
    model_df_modern.loc[
        (model_df_modern["year"] == 2026) &
        (
            model_df_modern["tournament"] ==
            "FIFA World Cup"
        )
    ].shape[0]
)

print(
    "\nCompleted 2026 World Cup rows before Elo filtering:",
    EXPECTED_AVAILABLE_TEST_MATCHES
)

print(
    "Completed 2026 World Cup rows after Elo filtering:",
    test_df.shape[0]
)

Train shape: (23192, 7)
Validation shape: (4106, 7)
Test shape: (102, 7)
Train years: 1994 - 2021
Validation years: 2022 - 2025
Test years: 2026 - 2026

Training target distribution:


match_result
home_win    0.484132
away_win    0.277208
draw        0.238660
Name: proportion, dtype: float64


Validation target distribution:


match_result
home_win    0.476376
away_win    0.292499
draw        0.231125
Name: proportion, dtype: float64


2026 World Cup test target distribution:


match_result
home_win    0.470588
away_win    0.294118
draw        0.235294
Name: proportion, dtype: float64


Completed 2026 World Cup rows before Elo filtering: 102
Completed 2026 World Cup rows after Elo filtering: 102


## Encoding and Scaling

Categorical predictors are one-hot encoded so they can be used in machine learning models. Numeric Elo-based features are standardized using the training data only, then the same transformation is applied to the test data. This prevents information from the test set from influencing preprocessing.


In [95]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder
)

feature_cols = (
    numeric_features +
    categorical_features
)

X_train = train_df[feature_cols].copy()
X_validation = validation_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_col].copy()
y_validation = validation_df[target_col].copy()
y_test = test_df[target_col].copy()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="drop"
)

# Fit transformations using training data only.
X_train_processed = preprocessor.fit_transform(
    X_train
)

# Apply the fitted transformations without refitting.
X_validation_processed = preprocessor.transform(
    X_validation
)

X_test_processed = preprocessor.transform(
    X_test
)

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(
    y_train
)

y_validation_encoded = label_encoder.transform(
    y_validation
)

y_test_encoded = label_encoder.transform(
    y_test
)

print(
    "X_train processed shape:",
    X_train_processed.shape
)

print(
    "X_validation processed shape:",
    X_validation_processed.shape
)

print(
    "X_test processed shape:",
    X_test_processed.shape
)

print(
    "Target classes:",
    list(label_encoder.classes_)
)

X_train processed shape: (23192, 100)
X_validation processed shape: (4106, 100)
X_test processed shape: (102, 100)
Target classes: ['away_win', 'draw', 'home_win']


In [96]:
feature_names = (
    preprocessor.get_feature_names_out()
)


def processed_array_to_df(
    array,
    columns,
    index
):
    return pd.DataFrame(
        (
            array.toarray()
            if hasattr(array, "toarray")
            else array
        ),
        columns=columns,
        index=index
    )


X_train_processed_df = processed_array_to_df(
    X_train_processed,
    feature_names,
    train_df.index
)

X_validation_processed_df = processed_array_to_df(
    X_validation_processed,
    feature_names,
    validation_df.index
)

X_test_processed_df = processed_array_to_df(
    X_test_processed,
    feature_names,
    test_df.index
)

y_train_df = pd.DataFrame(
    {
        "match_result": y_train.values,
        "match_result_encoded": y_train_encoded
    },
    index=train_df.index
)

y_validation_df = pd.DataFrame(
    {
        "match_result": y_validation.values,
        "match_result_encoded":
            y_validation_encoded
    },
    index=validation_df.index
)

y_test_df = pd.DataFrame(
    {
        "match_result": y_test.values,
        "match_result_encoded": y_test_encoded
    },
    index=test_df.index
)

X_train_processed_df.head()

,num__elo_diff,cat__tournament_ABCS Tournament,cat__tournament_AFC Asian Cup,cat__tournament_AFC Asian Cup qualification,cat__tournament_AFC Challenge Cup,cat__tournament_AFC Challenge Cup qualification,cat__tournament_AFC Solidarity Cup,cat__tournament_AFF Championship,cat__tournament_AFF Championship qualification,cat__tournament_African Cup of Nations,...,cat__tournament_USA Cup,cat__tournament_United Arab Emirates Friendship Tournament,cat__tournament_Unity Cup,cat__tournament_VFF Cup,cat__tournament_WAFF Championship,cat__tournament_Windward Islands Tournament,cat__neutral_False,cat__neutral_True,cat__is_world_cup_False,cat__is_world_cup_True
19503,-0.488182,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
19504,-0.219132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
19505,0.573474,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
19506,-1.611647,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
19507,-1.044461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0


In [97]:
# Save processed predictors.
X_train_processed_df.to_csv(
    PROCESSED_DIR / "X_train_processed_new.csv",
    index=False
)

X_validation_processed_df.to_csv(
    PROCESSED_DIR /
    "X_validation_processed_new.csv",
    index=False
)

X_test_processed_df.to_csv(
    PROCESSED_DIR / "X_test_processed_new.csv",
    index=False
)

# Save targets.
y_train_df.to_csv(
    PROCESSED_DIR / "y_train_new.csv",
    index=False
)

y_validation_df.to_csv(
    PROCESSED_DIR / "y_validation_new.csv",
    index=False
)

y_test_df.to_csv(
    PROCESSED_DIR / "y_test_new.csv",
    index=False
)

# Save readable splits for interpretation and heuristic models.
train_df.to_csv(
    PROCESSED_DIR /
    "train_matches_readable.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_DIR /
    "validation_matches_readable.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_DIR /
    "world_cup_2026_test_readable.csv",
    index=False
)

# Save metadata.
feature_names_df = pd.DataFrame({
    "feature_name": feature_names
})

label_mapping_df = pd.DataFrame({
    "match_result": label_encoder.classes_,
    "match_result_encoded":
        label_encoder.transform(
            label_encoder.classes_
        )
})

feature_names_df.to_csv(
    PROCESSED_DIR / "feature_names_new.csv",
    index=False
)

label_mapping_df.to_csv(
    PROCESSED_DIR / "label_mapping_new.csv",
    index=False
)

print(
    "Saved training, validation, and "
    "2026 World Cup test datasets."
)


Saved training, validation, and 2026 World Cup test datasets.


## Preprocessing Summary

- The International Football Results dataset was updated through July 15, 2026.
- The current data snapshot contains 102 completed 2026 FIFA World Cup matches; the third-place match and final are not included.
- Matches without valid dates or completed scores were removed.
- Legitimate scoreline outliers were retained after inspection.
- Team names were standardized before merging with Elo ratings.
- Elo ratings were matched to the most recent rating available strictly before each match date.
- The Elo source ends on December 13, 2025. Therefore, all 2026 World Cup matches use the latest available pre-2026 Elo ratings, preventing information from 2026 matches from influencing the held-out test predictors.
- The modeling dataset was restricted to matches from 1994 onward.
- Matches from 1994–2021 were assigned to training, matches from 2022–2025 to validation, and available completed 2026 FIFA World Cup matches to final testing.
- Only pre-match variables were retained as predictors.
- Elo differential was used rather than simultaneously including home Elo, away Elo, and their derived difference.
- Categorical variables were one-hot encoded and Elo differential was standardized using the training set only.
- The fitted preprocessing transformations were applied unchanged to validation and test data.